# 📦 Notebook 2 — Data Readiness and Production RAG Setup

This notebook validates the Product Finder data pack and provisions the default production retrieval path.

## Data files (pre-created in `product-finder/data/`)
| File | Purpose |
|------|---------|
| `products.csv` | 10 Syensqo SynPet product descriptions, categories, pH ranges, cautions |
| `compatibility_matrix.csv` | 15 compatibility pairs with verdicts, confidence, and safety rationale |

## What this notebook does
1. Verifies both data files exist in the `data/` folder
2. Validates schema, data types, and quality rules
3. Previews the product catalog and compatibility matrix
4. Provisions Blob Storage + Azure AI Search and indexes the data for RAG
5. Persists the stable RAG configuration into `azd env`

## Why this matters
Product Finder specialist agents now source retrieval context from Azure AI Search at runtime, making Search the single source of truth.

In [10]:
import sys, pathlib
import pandas as pd

def find_repo_root(start: pathlib.Path) -> pathlib.Path:
    cur = start.resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "shared" / "utils.py").exists() and (candidate / "workshop" / "product-finder").exists():
            return candidate
    raise RuntimeError("Could not locate repo root containing shared/utils.py and workshop/product-finder.")

repo_root = find_repo_root(pathlib.Path.cwd())
shared_dir = repo_root / "shared"
sys.path.insert(0, str(shared_dir))
import utils  # type: ignore

DATA_DIR = repo_root / "workshop" / "product-finder" / "data"
PRODUCTS_F = DATA_DIR / "products.csv"
COMPAT_F = DATA_DIR / "compatibility_matrix.csv"

utils.print_info(f"Data directory: {DATA_DIR}")
for f in [PRODUCTS_F, COMPAT_F]:
    if f.exists():
        utils.print_ok(f"Found: {f.name}")
    else:
        raise RuntimeError(f"Missing data file: {f}  — ensure product-finder/data/ files are present")

👉🏽 Data directory: C:\Users\sofiedelaet\Repos\ai-citadel-workshop\workshop\product-finder\data
✅ Found: products.csv ⌚ 17:26:35.554257 
✅ Found: compatibility_matrix.csv ⌚ 17:26:35.554370 


In [11]:
# ── Load and validate products ───────────────────────────────────────────────
products = pd.read_csv(PRODUCTS_F)
utils.print_info(f"Products loaded: {len(products)} rows")

req_product_cols = {"product_id","name","category","target_animal","age_group",
                    "condition","purpose","active_ingredients","ph_min","ph_max",
                    "description","caution"}
missing = req_product_cols - set(products.columns)
if missing:
    raise RuntimeError(f"Missing product columns: {sorted(missing)}")
utils.print_ok("Product schema valid")

dups = products[products.duplicated("product_id")]
if not dups.empty:
    raise RuntimeError(f"Duplicate product_ids: {dups['product_id'].tolist()}")
utils.print_ok("No duplicate product_ids")

# ── Load and validate compatibility matrix ───────────────────────────────────
compat = pd.read_csv(COMPAT_F)
utils.print_info(f"Compatibility pairs loaded: {len(compat)} rows")

req_compat_cols = {"product_a","product_b","compatibility","confidence","risk_tier","notes","recommendation"}
missing_c = req_compat_cols - set(compat.columns)
if missing_c:
    raise RuntimeError(f"Missing compatibility columns: {sorted(missing_c)}")
utils.print_ok("Compatibility schema valid")

invalid_conf = compat[(compat["confidence"] < 0) | (compat["confidence"] > 1)]
if not invalid_conf.empty:
    raise RuntimeError(f"Confidence values out of [0,1]: {invalid_conf[['product_a','product_b','confidence']].values.tolist()}")
utils.print_ok("All confidence values in valid range [0, 1]")

invalid_verdict = compat[~compat["compatibility"].isin(["compatible","incompatible","caution"])]
if not invalid_verdict.empty:
    raise RuntimeError(f"Invalid compatibility verdicts: {invalid_verdict['compatibility'].tolist()}")
utils.print_ok("All compatibility verdicts valid")

👉🏽 Products loaded: 10 rows
✅ Product schema valid ⌚ 17:26:45.016783 
✅ No duplicate product_ids ⌚ 17:26:45.020561 
👉🏽 Compatibility pairs loaded: 15 rows
✅ Compatibility schema valid ⌚ 17:26:45.023921 
✅ All confidence values in valid range [0, 1] ⌚ 17:26:45.025616 
✅ All compatibility verdicts valid ⌚ 17:26:45.026836 


In [12]:
# ── Preview data ─────────────────────────────────────────────────────────────
print("\n── PRODUCT CATALOG ─────────────────────────────────────────────────")
for _, row in products.iterrows():
    print(f"  {row['product_id']}  {row['name']:30s}  pH {row['ph_min']}-{row['ph_max']}  [{row['category']}]")

print("\n── COMPATIBILITY MATRIX ────────────────────────────────────────────")
for _, row in compat.iterrows():
    icon = "✅" if row["compatibility"] == "compatible" else ("❌" if row["compatibility"] == "incompatible" else "⚠️ ")
    print(f"  {icon}  {row['product_a']} + {row['product_b']}  →  {row['compatibility']:12s}  conf={row['confidence']:.2f}  [{row['risk_tier']}]")


── PRODUCT CATALOG ─────────────────────────────────────────────────
  SP001  SynPet Clean Pro                pH 6.5-7.0  [Shampoo]
  SP002  SynPet Gentle Care              pH 5.5-6.0  [Shampoo]
  SP003  SynPet Odor Shield              pH 7.0-7.5  [Cleanser]
  SP004  SynPet Flea Guard               pH 5.0-5.5  [Medicated]
  SP005  SynPet Coat Shine               pH 4.5-5.0  [Conditioner]
  SP006  SynPet Paw Care                 pH 5.5-6.5  [Topical]
  SP007  SynPet Puppy Fresh              pH 6.0-6.5  [Shampoo]
  SP008  SynPet Deep Clean               pH 8.0-8.5  [Cleanser]
  SP009  SynPet Dry Foam                 pH 6.0-7.0  [Waterless]
  SP010  SynPet Sensitive Plus           pH 5.5-6.0  [Hypoallergenic]

── COMPATIBILITY MATRIX ────────────────────────────────────────────
  ✅  SP001 + SP003  →  compatible    conf=0.94  [low]
  ✅  SP001 + SP005  →  compatible    conf=0.89  [low]
  ✅  SP001 + SP006  →  compatible    conf=0.96  [low]
  ✅  SP001 + SP009  →  compatible    conf=0.88  [lo

### Production RAG Setup: Blob + Azure AI Search

Specialist agents retrieve product and compatibility context from Azure AI Search at runtime:

1. Creates or reuses a Storage Account and uploads `products.csv` + `compatibility_matrix.csv` to Blob
2. Creates or reuses an Azure AI Search service and index
3. Indexes product and compatibility documents for retrieval
4. Saves the stable RAG settings into `azd env`
5. Later notebooks should re-read these settings from environment at runtime

In [17]:
# ── Provision Blob Storage and upload Product Finder data ─────────────────────
import json, pathlib, time

def find_repo_root(start: pathlib.Path) -> pathlib.Path:
    cur = start.resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "shared" / "utils.py").exists() and (candidate / "workshop" / "product-finder").exists():
            return candidate
    raise RuntimeError("Could not locate repo root containing shared/utils.py and workshop/product-finder.")

def run(cmd: str, ok_msg: str = "", fail_msg: str = ""):
    return utils.run(cmd, ok_msg, fail_msg)

def run_az_json(cmd: str, ok_msg: str, fail_msg: str):
    out = run(cmd, ok_msg, fail_msg)
    if not out.success:
        raise RuntimeError(f"{fail_msg}: {(out.text or '').strip()}")
    return out.json_data

def run_text(cmd: str, ok_msg: str = "", fail_msg: str = "") -> str:
    out = run(cmd, ok_msg, fail_msg)
    if not out.success:
        raise RuntimeError(f"{fail_msg or 'Command failed'}: {(out.text or '').strip()}")
    return (out.text or "").strip()

def set_azd_env(key: str, value: str):
    safe_value = value.replace('"', '\\"')
    out = run(f'azd env set {key} "{safe_value}"', "", f"Failed to persist azd env {key}")
    if not out.success:
        raise RuntimeError(f"Failed to persist azd env {key}: {(out.text or '').strip()}")

def get_azd_env(key: str) -> str:
    out = run(f"azd env get-value {key}", "", f"Missing azd env value [{key}]")
    if not out.success:
        raise RuntimeError(f"Missing azd env value [{key}]: {(out.text or '').strip()}")
    value = (out.text or "").strip()
    if not value or value.upper().startswith("ERROR:"):
        raise RuntimeError(f"Missing azd env value [{key}]: {value or 'empty value'}")
    return value

def get_azd_env_optional(key: str) -> str:
    out = run(f"azd env get-value {key}")
    if not out.success:
        return ""
    value = (out.text or "").strip()
    if value.upper().startswith("ERROR:"):
        return ""
    return value

def upload_blob_with_retry(storage_account: str, container_name: str, blob_name: str, file_path: pathlib.Path, attempts: int = 5):
    cmd = (
        f'az storage blob upload --account-name "{storage_account}" --container-name "{container_name}" '
        f'--name "{blob_name}" --file "{file_path}" --overwrite true --auth-mode login -o json'
    )
    for attempt in range(1, attempts + 1):
        out = run(cmd)
        if out.success:
            utils.print_ok(f"Uploaded {blob_name}")
            return
        if attempt < attempts:
            backoff = 5 * attempt
            utils.print_info(
                f"Upload retry for {blob_name} ({attempt}/{attempts - 1}) in {backoff}s..."
            )
            time.sleep(backoff)
    raise RuntimeError(f"Failed to upload {blob_name}: {(out.text or '').strip()}")

repo_root = find_repo_root(pathlib.Path.cwd())
data_dir = repo_root / "workshop" / "product-finder" / "data"
products_csv = data_dir / "products.csv"
compat_csv = data_dir / "compatibility_matrix.csv"
if not products_csv.exists() or not compat_csv.exists():
    raise RuntimeError("Data files not found. Run the first cell in this notebook first.")

sub_id = get_azd_env("AZURE_SUBSCRIPTION_ID")
rg = get_azd_env("SPOKE_RESOURCE_GROUP")
location = get_azd_env("AZURE_LOCATION")

existing_storage = get_azd_env_optional("PF_RAG_STORAGE_ACCOUNT")
existing_container = get_azd_env_optional("PF_RAG_CONTAINER")

# Env-first resolution: only reuse when storage account value exists and matches the expected prefix.
storage_prefix = "pfsarag"
create_storage = False
if existing_storage and existing_storage.lower().startswith(storage_prefix):
    storage_account = existing_storage.lower()
    utils.print_ok(f"Reusing storage account from env: {storage_account}")
else:
    timestamp_suffix = str(int(time.time()))
    storage_account = f"{storage_prefix}{timestamp_suffix}"
    create_storage = True
    utils.print_info(
        f"No valid env storage account with prefix '{storage_prefix}' found. "
        f"Will create one: {storage_account}"
    )

# 1) Create/reuse Storage Account
if create_storage:
    run_az_json(
        f'az storage account create -g "{rg}" -n "{storage_account}" -l "{location}" --sku Standard_LRS --min-tls-version TLS1_2 --allow-blob-public-access false -o json',
        f"Storage account created: {storage_account}",
        "Failed to create Storage account",
    )
else:
    utils.print_ok(f"Storage account reuse mode: {storage_account}")

# Keep RBAC flow consistent with specialist deployment notebooks: list -> create -> wait.
storage_scope = run_text(
    f'az storage account show -g "{rg}" -n "{storage_account}" --query id -o tsv',
    "",
    "Failed to resolve storage account scope",
)
acct = run_az_json("az account show -o json", "", "Failed to resolve current Azure account")
principal_name = str(acct.get("user", {}).get("name", "")).strip()
principal_kind = str(acct.get("user", {}).get("type", "")).lower()

if principal_kind == "user":
    principal_object_id = run_text(
        "az ad signed-in-user show --query id -o tsv",
        "",
        "Failed to resolve signed-in user object id",
    )
    principal_type_arg = "User"
elif principal_kind == "serviceprincipal":
    principal_object_id = run_text(
        f'az ad sp show --id "{principal_name}" --query id -o tsv',
        "",
        "Failed to resolve signed-in service principal object id",
    )
    principal_type_arg = "ServicePrincipal"
else:
    raise RuntimeError(f"Unsupported account type for RBAC assignment: {principal_kind}")

existing_role = run_text(
    f'az role assignment list --assignee-object-id "{principal_object_id}" --role "Storage Blob Data Contributor" --scope "{storage_scope}" --query "[0].id" -o tsv',
    "",
    "Failed to query role assignments",
)
if existing_role:
    utils.print_ok("Storage Blob Data Contributor role already assigned")
else:
    run_az_json(
        f'az role assignment create --assignee-object-id "{principal_object_id}" --assignee-principal-type {principal_type_arg} --role "Storage Blob Data Contributor" --scope "{storage_scope}" -o json',
        "Granted Storage Blob Data Contributor role",
        "Failed to grant Storage Blob Data Contributor role",
    )

# 2) Container handling: create only when storage account is newly created.
if create_storage:
    container_name = "product-finder-data"
    run_az_json(
        f'az storage container create --account-name "{storage_account}" --name "{container_name}" --auth-mode login -o json',
        f"Blob container created: {container_name}",
        "Failed to create Blob container",
    )
else:
    container_name = existing_container
    if not container_name:
        raise RuntimeError(
            "Reusing storage account but PF_RAG_CONTAINER is not set in env. "
            "Set PF_RAG_CONTAINER or create a new storage account run."
        )
    utils.print_ok(f"Reusing container from env: {container_name}")

# 3) Upload source files to the selected container.
upload_blob_with_retry(storage_account, container_name, "products.csv", products_csv)
upload_blob_with_retry(storage_account, container_name, "compatibility_matrix.csv", compat_csv)

blob_base = f"https://{storage_account}.blob.core.windows.net/{container_name}"
set_azd_env("PF_RAG_STORAGE_ACCOUNT", storage_account)
set_azd_env("PF_RAG_CONTAINER", container_name)
set_azd_env("PF_RAG_SOURCE_PRODUCTS_BLOB", f"{blob_base}/products.csv")
set_azd_env("PF_RAG_SOURCE_COMPAT_BLOB", f"{blob_base}/compatibility_matrix.csv")
utils.print_ok("Blob storage and source data are ready.")

⚙️ Running: azd env get-value AZURE_SUBSCRIPTION_ID 
✅  ⌚ 17:59:02.468837 :0s]
⚙️ Running: azd env get-value SPOKE_RESOURCE_GROUP 
✅  ⌚ 17:59:03.004438 :0s]
⚙️ Running: azd env get-value AZURE_LOCATION 
✅  ⌚ 17:59:03.524273 :0s]
⚙️ Running: azd env get-value PF_RAG_STORAGE_ACCOUNT 
⚙️ Running: azd env get-value PF_RAG_CONTAINER 
✅ Reusing storage account from env: pfsarag1781020480 ⌚ 17:59:04.566256 
✅ Storage account reuse mode: pfsarag1781020480 ⌚ 17:59:04.567093 
⚙️ Running: az storage account show -g "rg-citadel-workshop-spoke-1" -n "pfsarag1781020480" --query id -o tsv 
✅  ⌚ 17:59:07.873117 :3s]
⚙️ Running: az account show -o json 
✅  ⌚ 17:59:09.441817 :1s]
⚙️ Running: az ad signed-in-user show --query id -o tsv 
✅  ⌚ 17:59:12.665516 :3s]
⚙️ Running: az role assignment list --assignee-object-id "0588da48-0fda-4940-88e7-0d5aa1ea9208" --role "Storage Blob Data Contributor" --scope "/subscriptions/b881797f-b05b-4743-8516-17a967f31841/resourceGroups/rg-citadel-workshop-spoke-1/provide

### Azure AI Search Deployment and Indexing

Creates or reuses the Search service and index, uploads documents, and persists Search settings to `azd env`.

In [ ]:
# ── Provision Azure AI Search and index Product Finder data ───────────────────
# This cell assumes the previous storage cell has completed successfully.
import requests
import pandas as pd

existing_search_service = get_azd_env_optional("PF_RAG_SEARCH_SERVICE")
existing_index = get_azd_env_optional("PF_RAG_INDEX")

# Env-first resolution for Search service: reuse only when name matches expected prefix.
search_prefix = "pf-rag-"
if existing_search_service and existing_search_service.lower().startswith(search_prefix):
    search_service = existing_search_service.lower()
    utils.print_ok(f"Reusing Search service from env: {search_service}")
else:
    timestamp_suffix = str(int(time.time()))
    search_service = f"{search_prefix}{timestamp_suffix}"
    utils.print_info(
        f"No valid env Search service with prefix '{search_prefix}' found. "
        f"Will create one: {search_service}"
    )

index_name = existing_index or "pf-products-index"

utils.print_info(f"RAG search service:  {search_service}")
utils.print_info(f"RAG index:           {index_name}")

search_resource_id = f"/subscriptions/{sub_id}/resourceGroups/{rg}/providers/Microsoft.Search/searchServices/{search_service}"
search_show = run(f'az resource show --ids "{search_resource_id}" -o json')
if not search_show.success:
    create_body = {
        "location": location,
        "sku": {"name": "basic"},
        "properties": {"replicaCount": 1, "partitionCount": 1, "hostingMode": "default"},
    }
    body_escaped = json.dumps(create_body).replace('"', '\\"')
    run_az_json(
        f'az rest --method put --url "https://management.azure.com{search_resource_id}?api-version=2023-11-01" --body "{body_escaped}" -o json',
        f"Search service create requested: {search_service}",
        "Failed to create Azure AI Search service",
    )
else:
    utils.print_ok(f"Search service exists: {search_service}")

poll_started = time.time()
poll_timeout_s = 30
for attempt in range(1, 41):
    if time.time() - poll_started > poll_timeout_s:
        raise RuntimeError(f"Timed out waiting for Search service provisioning after {poll_timeout_s}s")
    svc = run_az_json(
        f'az resource show --ids "{search_resource_id}" -o json',
        "",
        "Failed to query Search service status",
    )
    prov = str(svc.get("properties", {}).get("provisioningState", "")).lower()
    utils.print_info(f"Search provisioning state check {attempt}/40: {prov or 'unknown'}")
    if prov == "succeeded":
        utils.print_ok("Search service provisioning succeeded")
        break
    time.sleep(10)

keys = run_az_json(
    f'az rest --method post --url "https://management.azure.com{search_resource_id}/listAdminKeys?api-version=2023-11-01" -o json',
    "Retrieved Search admin keys",
    "Failed to retrieve Search admin keys",
)
search_admin_key = keys.get("primaryKey")
if not search_admin_key:
    raise RuntimeError("Search admin key not found in listAdminKeys response")

search_endpoint = f"https://{search_service}.search.windows.net"
headers = {"Content-Type": "application/json", "api-key": search_admin_key}

index_schema = {
    "name": index_name,
    "fields": [
        {"name": "id", "type": "Edm.String", "key": True, "searchable": False, "filterable": True, "sortable": True},
        {"name": "entity_type", "type": "Edm.String", "searchable": False, "filterable": True, "facetable": True},
        {"name": "source_file", "type": "Edm.String", "searchable": False, "filterable": True},
        {"name": "product_id", "type": "Edm.String", "searchable": True, "filterable": True},
        {"name": "product_a", "type": "Edm.String", "searchable": True, "filterable": True},
        {"name": "product_b", "type": "Edm.String", "searchable": True, "filterable": True},
        {"name": "compatibility", "type": "Edm.String", "searchable": True, "filterable": True, "facetable": True},
        {"name": "risk_tier", "type": "Edm.String", "searchable": False, "filterable": True, "facetable": True},
        {"name": "confidence", "type": "Edm.Double", "searchable": False, "filterable": True, "sortable": True},
        {"name": "title", "type": "Edm.String", "searchable": True},
        {"name": "text", "type": "Edm.String", "searchable": True},
    ],
}
idx_resp = requests.put(f"{search_endpoint}/indexes/{index_name}?api-version=2024-07-01", headers=headers, json=index_schema, timeout=90)
if idx_resp.status_code not in (200, 201):
    raise RuntimeError(f"Failed to create/update index: {idx_resp.status_code} {idx_resp.text}")
utils.print_ok(f"Search index ready: {index_name}")

products_df = pd.read_csv(products_csv).fillna("")
compat_df = pd.read_csv(compat_csv).fillna("")

docs = []
for _, row in products_df.iterrows():
    pid = str(row.get("product_id", "")).strip()
    docs.append({
        "id": f"product-{pid}",
        "entity_type": "product",
        "source_file": "products.csv",
        "product_id": pid,
        "title": str(row.get("name", "")),
        "text": (
            f"Product {row.get('name', '')} ({pid}). "
            f"Category: {row.get('category', '')}. "
            f"Target animal: {row.get('target_animal', '')}. "
            f"Age group: {row.get('age_group', '')}. "
            f"Condition: {row.get('condition', '')}. "
            f"Purpose: {row.get('purpose', '')}. "
            f"Active ingredients: {row.get('active_ingredients', '')}. "
            f"pH: {row.get('ph_min', '')}-{row.get('ph_max', '')}. "
            f"Description: {row.get('description', '')}. "
            f"Caution: {row.get('caution', '')}."
        ),
    })

for i, row in compat_df.iterrows():
    conf = row.get("confidence", "")
    try:
        conf = float(conf)
    except Exception:
        conf = None
    docs.append({
        "id": f"compat-{i}",
        "entity_type": "compatibility",
        "source_file": "compatibility_matrix.csv",
        "product_a": str(row.get("product_a", "")),
        "product_b": str(row.get("product_b", "")),
        "compatibility": str(row.get("compatibility", "")),
        "risk_tier": str(row.get("risk_tier", "")),
        "confidence": conf,
        "title": f"Compatibility {row.get('product_a', '')} + {row.get('product_b', '')}",
        "text": (
            f"Compatibility check between {row.get('product_a', '')} and {row.get('product_b', '')}. "
            f"Verdict: {row.get('compatibility', '')}. "
            f"Confidence: {row.get('confidence', '')}. "
            f"Risk tier: {row.get('risk_tier', '')}. "
            f"Notes: {row.get('notes', '')}. "
            f"Recommendation: {row.get('recommendation', '')}."
        ),
    })

batch = {"value": [{"@search.action": "upload", **d} for d in docs]}
push_resp = requests.post(f"{search_endpoint}/indexes/{index_name}/docs/index?api-version=2024-07-01", headers=headers, json=batch, timeout=120)
if push_resp.status_code not in (200, 201):
    raise RuntimeError(f"Failed to upload documents to index: {push_resp.status_code} {push_resp.text}")

result = push_resp.json()
failed = [r for r in result.get("value", []) if not r.get("status", False)]
if failed:
    raise RuntimeError(f"Indexing completed with failures: {failed[:3]}")
utils.print_ok(f"Indexed {len(docs)} documents into {index_name}")

set_azd_env("PF_RAG_ENABLED", "true")
set_azd_env("PF_RAG_SEARCH_SERVICE", search_service)
set_azd_env("PF_RAG_SEARCH_ENDPOINT", search_endpoint)
set_azd_env("PF_RAG_INDEX", index_name)
set_azd_env("PF_RAG_SEARCH_API_KEY", search_admin_key)
utils.print_ok("Azure AI Search resources are ready and saved to azd env")
utils.print_info("Specialist agents will now use Azure AI Search at runtime (no embedded CSV strings).")

⚙️ Running: azd env get-value PF_RAG_SEARCH_SERVICE 
⚙️ Running: azd env get-value PF_RAG_INDEX 
👉🏽 No valid env Search service with prefix 'pf-rag-' found. Will create one: pf-rag-1781021163
👉🏽 RAG search service:  pf-rag-1781021163
👉🏽 RAG index:           pf-products-index
⚙️ Running: az resource show --ids "/subscriptions/b881797f-b05b-4743-8516-17a967f31841/resourceGroups/rg-citadel-workshop-spoke-1/providers/Microsoft.Search/searchServices/pf-rag-1781021163" -o json 
⚙️ Running: az rest --method put --url "https://management.azure.com/subscriptions/b881797f-b05b-4743-8516-17a967f31841/resourceGroups/rg-citadel-workshop-spoke-1/providers/Microsoft.Search/searchServices/pf-rag-1781021163?api-version=2023-11-01" --body "{\"location\": \"swedencentral\", \"sku\": {\"name\": \"basic\"}, \"properties\": {\"replicaCount\": 1, \"partitionCount\": 1, \"hostingMode\": \"default\"}}" -o json 
✅ Search service create requested: pf-rag-1781021163 ⌚ 18:06:16.644713 :8s]
⚙️ Running: az resource 

KeyboardInterrupt: 